In [1]:
from pathlib import Path
import re
import pandas as pd

# Folder containing the importance-magnitude CSV files
input_folder = Path(".")

# Match files such as:
filename_pattern = re.compile(
    r"^importance_magnitude_cnn1_RfxCas13a_validation_(\d+)\.csv$"
)

# Detect matching files
matching_files = []

for file_path in input_folder.glob(
    "importance_magnitude_cnn1_RfxCas13a_validation_*.csv"
):
    match = filename_pattern.match(file_path.name)

    if match:
        model_number = int(match.group(1))
        matching_files.append((model_number, file_path))

# Sort by model number
matching_files.sort(key=lambda x: x[0])

print(f"Detected {len(matching_files)} importance-magnitude files:")

for model_number, file_path in matching_files:
    print(f"Model {model_number}: {file_path.name}")

if len(matching_files) == 0:
    raise FileNotFoundError(
        "No matching importance-magnitude CSV files were found."
    )

if len(matching_files) != 10:
    print(
        f"\nWarning: Expected 10 files, but detected "
        f"{len(matching_files)} files."
    )


# Read and combine all files
all_importance_tables = []

for model_number, file_path in matching_files:

    df = pd.read_csv(file_path)

    # Check that the expected columns exist
    required_columns = {"Feature", "Importance Magnitude"}

    if not required_columns.issubset(df.columns):
        raise ValueError(
            f"{file_path.name} does not contain the required columns: "
            f"'Feature' and 'Importance Magnitude'."
        )

    # Keep only the required columns
    df = df[["Feature", "Importance Magnitude"]].copy()

    # Convert importance values to numeric
    df["Importance Magnitude"] = pd.to_numeric(
        df["Importance Magnitude"],
        errors="coerce"
    )

    # Add model number for tracking
    df["Model Number"] = model_number

    all_importance_tables.append(df)

# Combine all model tables
combined_df = pd.concat(
    all_importance_tables,
    ignore_index=True
)

# Calculate average importance magnitude for each feature
average_importance = (
    combined_df
    .groupby("Feature", as_index=False)
    .agg(
        Average_Importance_Magnitude=(
            "Importance Magnitude",
            "mean"
        ),
        Number_of_Models=(
            "Importance Magnitude",
            "count"
        )
    )
)

# Sort from highest to lowest average importance
average_importance = average_importance.sort_values(
    by="Average_Importance_Magnitude",
    ascending=False,
    ignore_index=True
)

# Save the final table
output_file = (
    input_folder /
    "average_importance_magnitude_cnn1_RfxCas13a_validation.csv"
)

average_importance.to_csv(output_file, index=False)

print(f"\nSaved: {output_file.name}")
print("\nAverage importance magnitude table:")
print(average_importance)

Detected 10 importance-magnitude files:
Model 9: importance_magnitude_cnn1_RfxCas13a_validation_9.csv
Model 22: importance_magnitude_cnn1_RfxCas13a_validation_22.csv
Model 41: importance_magnitude_cnn1_RfxCas13a_validation_41.csv
Model 42: importance_magnitude_cnn1_RfxCas13a_validation_42.csv
Model 56: importance_magnitude_cnn1_RfxCas13a_validation_56.csv
Model 73: importance_magnitude_cnn1_RfxCas13a_validation_73.csv
Model 76: importance_magnitude_cnn1_RfxCas13a_validation_76.csv
Model 83: importance_magnitude_cnn1_RfxCas13a_validation_83.csv
Model 88: importance_magnitude_cnn1_RfxCas13a_validation_88.csv
Model 98: importance_magnitude_cnn1_RfxCas13a_validation_98.csv

Saved: average_importance_magnitude_cnn1_RfxCas13a_validation.csv

Average importance magnitude table:
     Feature  Average_Importance_Magnitude  Number_of_Models
0    Pos56_C                      0.009281                10
1    Pos54_G                      0.009113                10
2    Pos56_G                      0

In [11]:
import pandas as pd

# Read the average importance-magnitude CSV file
df = pd.read_csv("average_importance_magnitude_cnn1_RfxCas13a_validation.csv")

# Extract position and base from feature names
extracted = df["Feature"].str.extract(r"^Pos(\d+)_([ACGU])$")

df["Position"] = pd.to_numeric(extracted[0], errors="coerce")
df["Base"] = extracted[1]

# Ensure importance magnitudes are numeric
df["Average_Importance_Magnitude"] = pd.to_numeric(
    df["Average_Importance_Magnitude"],
    errors="coerce"
)

# Remove rows that don't match the expected format
df = df.dropna(
    subset=["Position", "Base", "Average_Importance_Magnitude"]
).copy()

df["Position"] = df["Position"].astype(int)

# Define windows (using actual positions)
windows = {
    "First_6": (37, 42),
    "Middle_18": (43, 60),
    "Last_6": (61, 66)
}

bases = ["A", "C", "G", "U"]

results = []

for base in bases:
    base_df = df[df["Base"] == base]

    row = {"Base": base}

    for window_name, (start, end) in windows.items():
        avg = base_df.loc[
            base_df["Position"].between(start, end),
            "Average_Importance_Magnitude"
        ].mean()

        row[window_name] = avg

    results.append(row)

# Convert to DataFrame
result_df = pd.DataFrame(results)

# Save as CSV
result_df.to_csv(
    "average_importance_magnitude_by_Base_position_range_cnn1_RfxCas13a_validation.csv",
    index=False
)

print(result_df)

  Base   First_6  Middle_18    Last_6
0    A  0.004286   0.005158  0.003221
1    C  0.003487   0.005497  0.003492
2    G  0.004678   0.006384  0.004016
3    U  0.003937   0.005012  0.003050
